Проверяем, как перенос ворот с 30 на 40 уровень влияет на удержание игроков. Контроль gate_30 (изначально), тест gate_40. Главная метрика retention_7 (вернулся ли через 7 дней, так как главное удержание). Вспомогательные retention_1 и sum_gamerounds. H0: удержание на 7 день одинаковое. H1: удержание на 7 день отличается. α пусть будет стандартный 0.05

In [ ]:
import pandas as pd
from scipy.stats import chisquare
import sqlite3
from scipy.stats import chi2_contingency
import numpy as np
from scipy.stats import mannwhitneyu

In [ ]:
df = pd.read_csv('cookie_cats.csv')
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90189 entries, 0 to 90188
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   userid          90189 non-null  int64 
 1   version         90189 non-null  object
 2   sum_gamerounds  90189 non-null  int64 
 3   retention_1     90189 non-null  bool  
 4   retention_7     90189 non-null  bool  
dtypes: bool(2), int64(2), object(1)
memory usage: 2.2+ MB


,userid,sum_gamerounds
count,9.018900e+04,90189.000000
mean,4.998412e+06,51.872457
std,2.883286e+06,195.050858
min,1.160000e+02,0.000000
25%,2.512230e+06,5.000000
50%,4.995815e+06,16.000000
75%,7.496452e+06,51.000000
max,9.999861e+06,49854.000000


Тут странно, что кто-то сыграл почти 50000 раундов за 14 дней, скорее всего ошибка или бот. Стоит учитывать, что часть игроков не сыграла ни одного раунда. Ну и среднее высокое относительно медианы, есть небольшое количество очень активных игроков, которые тянут вверх

In [ ]:
print('дублей:',sum(df.duplicated('userid')))
f_obs = df.value_counts('version')
print(f_obs)
print(chisquare(f_obs))
print()
print(df.nlargest(10, 'sum_gamerounds'))
print('доля нулей',(df['sum_gamerounds'] == 0).mean())

дублей: 0
version
gate_40    45489
gate_30    44700
Name: count, dtype: int64
Power_divergenceResult(statistic=np.float64(6.9024049496058275), pvalue=np.float64(0.008607987810836262))

        userid  version  sum_gamerounds  retention_1  retention_7
57702  6390605  gate_30           49854        False         True
7912    871500  gate_30            2961         True         True
29417  3271615  gate_40            2640         True        False
43671  4832608  gate_30            2438         True         True
48188  5346171  gate_40            2294         True         True
46344  5133952  gate_30            2251         True         True
87007  9640085  gate_30            2156         True         True
36933  4090246  gate_40            2124         True         True
88328  9791599  gate_40            2063         True         True
6536    725080  gate_40            2015         True         True
доля нулей 0.0442847797403231


Дублей нет, p-value < α, разница между группами слишком большая, чтобы списать на случайность, учитывать как ограничение. Сыгранных раундов 49854 у 1 игрока, у остальных не больше 3000, явная ошибка, удалю

In [ ]:
df = df[df['sum_gamerounds'] < 10000]
len(df)

90188

In [ ]:
con = sqlite3.connect(':memory:')
df.to_sql('players', con, index=False)
query = '''select version, count(userid) as players ,
  round(avg(retention_1)*100, 2) as retention_1_pct,
  round(avg(retention_7)*100,2) as retention_7_pct,
  round(avg(sum_gamerounds),2) as gamerounds_avg
  from players group by version'''
pd.read_sql(query, con)


,version,players,retention_1_pct,retention_7_pct,gamerounds_avg
0,gate_30,44699,44.82,19.02,51.34
1,gate_40,45489,44.23,18.20,51.30


In [ ]:
pvalues = {}
for metric in ['retention_1', 'retention_7']:
    table = pd.crosstab(df['version'], df[metric])
    p = chi2_contingency(table)
    pvalues[metric] = p.pvalue
    print(metric, 'p-value =', round(p.pvalue, 4))

retention_1 p-value = 0.075
retention_7 p-value = 0.0016


Перенос ворот снижает удержание 7 дня. Это логично т. к. большинство доходит до них не за 1 день

Зафиксировали сид

In [ ]:
rng = np.random.default_rng(42)

In [ ]:
for metric in ['retention_1', 'retention_7']:
    a = df[df['version'] == 'gate_30'][metric].to_numpy()
    b = df[df['version'] == 'gate_40'][metric].to_numpy()
    diffs = []
    for i in range(5000):
        sample_a = rng.choice(a, size=len(a), replace=True)
        gate30_mean = sample_a.mean()
        sample_b = rng.choice(b, size=len(b), replace=True)
        gate40_mean = sample_b.mean()
        dif = gate30_mean - gate40_mean
        diffs.append(dif)
    lo, hi = np.percentile(diffs, [2.5, 97.5]) * 100
    print(f'{metric}: 95% ДИ [{lo:+.2f}; {hi:+.2f}] п.п.')

retention_1: 95% ДИ [-0.05; +1.23] п.п.
retention_7: 95% ДИ [+0.32; +1.33] п.п.


Так как разница плюсовая для retention_7, то перенос ворот ухудшает удержание. Для retention_1 интервал задевает 0, разницы может и не быть

In [ ]:
df.groupby('version')['sum_gamerounds'].median()

,sum_gamerounds
version,
gate_30,17.0
gate_40,16.0


In [ ]:
a = df[df['version'] == 'gate_30']['sum_gamerounds'].to_numpy()
b = df[df['version'] == 'gate_40']['sum_gamerounds'].to_numpy()

In [ ]:
pvalues['sum_gamerounds'] = mannwhitneyu(a, b).pvalue
pvalues['sum_gamerounds']

np.float64(0.05089155279145376)

На то, сколько игроки играют, расположение ворот значимого влияния не оказывает. А влияет на то, возвращаются ли они на 7 день.

In [ ]:
grp = df.groupby('version')
r1 = grp['retention_1'].mean() * 100
r7 = grp['retention_7'].mean() * 100
med = grp['sum_gamerounds'].median()
p_list = [pvalues['retention_1'], pvalues['retention_7'], pvalues['sum_gamerounds']]
summary = pd.DataFrame({
    'Метрика': ['retention_1', 'retention_7', 'Медиана раундов'],
    'gate_30': [f"{r1['gate_30']:.2f}%", f"{r7['gate_30']:.2f}%", med['gate_30']],
    'gate_40': [f"{r1['gate_40']:.2f}%", f"{r7['gate_40']:.2f}%", med['gate_40']],
    'Разница': [f"{r1['gate_30'] - r1['gate_40']:+.2f} п.п.",
                f"{r7['gate_30'] - r7['gate_40']:+.2f} п.п.",
                f"{med['gate_30'] - med['gate_40']:+.0f} раунд"],
    'p-value': [round(p, 4) for p in p_list],
    'Вывод': ['значимо' if p < 0.05 else 'не значимо' for p in p_list],
})
summary

,Метрика,gate_30,gate_40,Разница,p-value,Вывод
0,retention_1,44.82%,44.23%,+0.59 п.п.,0.0750,не значимо
1,retention_7,19.02%,18.20%,+0.82 п.п.,0.0016,значимо
2,Медиана раундов,17.0,16.0,+1 раунд,0.0509,не значимо


**Выводы**

Retention_7 упала (на 0,82 п.п., по бутстрапу от 0,3 до 1,3 п.п., p = 0,0016).

Retention_1 значительно не изменилась (p = 0,075). На то, сколько игроки играют, расположение ворот значимо не влияет (p = 0,051).

Ворота лучше оставить на 30 уровне, так как главная метрика падает.

Если ворота на 30 уровне, то я предполагаю, что игроку ещё не хватило и хочется больше. А на 40 игроку уже хватило, и он уйдёт.

Ограничение: выборка перекошена по размеру групп (SRM, p = 0,009). В рабочей ситуации нужно узнать у разработчиков, почему так произошло: группы могут быть неодинаковые по составу, и разница уже будет вызвана не переносом ворот.